# 第2周：系统管理

> **学习目标**：理解 systemd 原理与 Unit/Timer/Journald、用户组与 sudo 配置、PAM 认证机制、存储管理（LVM/fstab）、包管理深入

---

## 开篇：系统管理员的工具箱

第一周我们深入了用户空间。这周切换到**系统管理员**的视角：

- **如何让一个程序开机自启？**（systemd service）
- **如何定时执行任务？**（systemd timer）
- **日志去哪了？**（journald）
- **怎么管理用户权限？**（useradd / sudo / PAM）
- **磁盘不够了怎么办？**（LVM 在线扩容）

这周的内容非常实用——几乎每一条知识都会在你维护服务器时用到。

---

## Day 8：systemd 原理与 Unit 文件

systemd 是 Linux 系统的 **PID 1**——它是内核启动的第一个用户空间进程。

### Unit 类型

| Unit 类型 | 后缀 | 用途 |
|-----------|------|------|
| service | `.service` | 管理一个服务/程序 |
| timer | `.timer` | 定时任务（现代化 cron） |
| socket | `.socket` | 监听的 socket，按需启动服务 |
| target | `.target` | 一组 unit 的分组（类似运行级别） |
| mount | `.mount` | 文件系统挂载点 |
| slice | `.slice` | 资源管理分组（配合 cgroup） |

### Service Unit 文件结构

```ini
[Unit]
Description=My Python Web App
After=network.target
Wants=redis.service

[Service]
Type=simple
User=www
WorkingDirectory=/opt/myapp
ExecStart=/opt/myapp/venv/bin/python app.py
Restart=on-failure
RestartSec=5
Environment="APP_ENV=production"

[Install]
WantedBy=multi-user.target
```

Type 的区别：
- `simple`：ExecStart 启动后 systemd 就认为服务已就绪（默认）
- `forking`：进程 fork 到后台后 systemd 才认为就绪（传统 daemon）
- `oneshot`：执行一次就退出，不长时间运行

In [ ]:
# 查看系统上的 unit
! echo "列出所有 service unit："
! systemctl list-units --type=service --all | head -20
! echo ""
! echo "查看 sshd 服务配置："
! systemctl cat sshd.service 2>/dev/null | head -15 || echo "sshd.service 配置不可读"
! echo ""
! echo "查看服务状态："
! systemctl status sshd.service 2>/dev/null | head -10 || echo "sshd 未安装"

In [ ]:
# 创建一个自定义 systemd service（演示）
! cat << 'UNIT'
[Unit]
Description=Hello Timer Demo Service

[Service]
Type=oneshot
ExecStart=/bin/bash -c 'echo "Hello from systemd: $(date)"'

[Install]
WantedBy=multi-user.target
UNIT
! echo ""
! echo "在真实系统中执行："
! echo "  sudo tee /etc/systemd/system/hello.service"
! echo "  sudo systemctl daemon-reload"
! echo "  sudo systemctl start hello"
! echo "  journalctl -u hello"

---

## Day 9：systemd Timer 与日志管理

### Timer — 替代 cron 的现代化定时任务

优势：
1. 可以跨系统重启持久化（`Persistent=true`）
2. 可以设定随机延迟（`RandomizedDelaySec`）
3. 日志统一管理（自动进 journald）
4. 开机错过时间后自动补跑

```ini
# /etc/systemd/system/check.timer
[Unit]
Description=Run hourly system check

[Timer]
OnCalendar=hourly
Persistent=true
RandomizedDelaySec=60

[Install]
WantedBy=timers.target
```

OnCalendar 格式：`minutely` / `hourly` / `daily` / `Mon..Fri 09:00:00` / `*-*-01 02:00:00` / `Sun 03:00:00`

In [ ]:
# 创建 timer unit（演示文件）
! cat << 'TIMER' > /tmp/demo.timer
[Unit]
Description=Run demo every minute

[Timer]
OnCalendar=minutely
Persistent=true

[Install]
WantedBy=timers.target
TIMER
! cat /tmp/demo.timer
! rm -f /tmp/demo.timer

In [ ]:
# journalctl 查询演示
! echo "查看 journald 日志占用空间："
! journalctl --disk-usage 2>/dev/null || echo "需要系统有 journald"
! echo ""
! echo "查看本次启动以来的错误日志（前 10 条）："
! journalctl -p err -b --no-pager 2>/dev/null | head -10 || echo "无错误日志或无权限"
! echo ""
! echo "日志管理："
! echo "  journalctl --vacuum-time=7d    # 只保留最近 7 天"
! echo "  journalctl --vacuum-size=500M  # 只保留 500MB"

---

## Day 10：用户、组与 sudo

### 用户管理的三个关键文件

| 文件 | 内容 |
|------|------|
| `/etc/passwd` | 用户账号信息（login、UID、GID、home、shell） |
| `/etc/shadow` | 加密密码和过期策略（仅 root 可读） |
| `/etc/group` | 组信息（组名、GID、成员列表） |

### 核心命令

```bash
# 创建系统用户（不可登录）
sudo useradd -r -s /usr/sbin/nologin -d /opt/myapp myapp

# 加入附加组
sudo usermod -aG docker alice

# sudo 权限配置（用 visudo！）
# %deployers ALL=(root) NOPASSWD: /usr/bin/systemctl restart *
```

In [ ]:
# 探索用户信息
! echo "当前用户: $(whoami)"
! echo "所属组: $(id -Gn)"
! echo ""
! grep "^$(whoami):" /etc/passwd 2>/dev/null || echo "passwd 条目不可读"
! echo ""
! head -5 /etc/group
! echo ""
! echo "sudo 配置必须用 visudo 编辑！"
! echo "建议放在 /etc/sudoers.d/ 下单独文件"

In [ ]:
# 创建应用用户（演示）
! sudo useradd -r -s /usr/sbin/nologin -M apprunner 2>/dev/null && \
  echo "用户 apprunner 已创建" || echo "用户已存在或无权创建"
! id apprunner 2>/dev/null || echo "apprunner 不存在"
! grep apprunner /etc/passwd 2>/dev/null || true

---

## Day 11：PAM 认证

PAM = Pluggable Authentication Modules，提供统一的认证接口。

所有需要认证的程序（login、sshd、sudo、passwd）都通过 PAM 处理，配置在 `/etc/pam.d/` 下。

**控制类型**：
| 类型 | 含义 |
|------|------|
| `required` | 必须通过。失败不影响其他模块执行 |
| `requisite` | 必须通过。失败立即返回 |
| `sufficient` | 通过且前面没有 required 失败，立即返回成功 |
| `optional` | 可选，不影响认证结果 |

**常用模块**：
- `pam_unix.so`：验证 /etc/shadow 密码
- `pam_tally2.so`：登录失败计数和锁定
- `pam_limits.so`：读取 `/etc/security/limits.conf`

In [ ]:
# 查看 PAM 配置
! echo "PAM 配置目录："
! ls /etc/pam.d/ 2>/dev/null | head -10 || echo "目录不可读"
! echo ""
! echo "当前用户的资源限制："
! ulimit -a | head -8
! echo ""
! echo "limits.conf 配置示例："
! echo "  @deployers    hard    nproc       100"
! echo "  @deployers    hard    nofile      65536"
! echo "  *             soft    core        0"

---

## Day 12：存储管理

### 磁盘分区 → 文件系统 → 挂载

```
磁盘 (sda) → 分区 (sda1, sda2) → 文件系统 (ext4) → 挂载点 (/, /home)
```

### LVM — 逻辑卷管理

LVM 将物理磁盘抽象为资源池，实现灵活的存储管理：

```
PV（物理卷）→ VG（卷组）→ LV（逻辑卷）→ 文件系统 → 挂载
/dev/sda1 ──▶        ──▶ lv_root (/)
/dev/sdb1 ──▶  vg0   ──▶ lv_data (/data)
               池     ──▶ lv_backup (/backup)
```

**LVM 的杀手级特性**：
- **在线扩容**：LV 可以直接增大，不需要卸载
- **快照**：创建 LV 的只读快照，用于备份
- **迁移**：数据从一个 PV 移到另一个 PV，零停机

In [ ]:
# 查看当前磁盘和分区状态
! echo "块设备列表："
! lsblk
! echo ""
! echo "文件系统空间："
! df -h
! echo ""
! echo "查看 UUID："
! blkid 2>/dev/null | head -5 || echo "需要 root 权限"

In [ ]:
# 用回环设备模拟磁盘管理
! dd if=/dev/zero of=/tmp/vdisk.img bs=1M count=256 2>/dev/null
! echo "虚拟磁盘文件已创建: 256MB"
! mkfs.ext4 /tmp/vdisk.img 2>/dev/null
! echo "已格式化为 ext4"
! file /tmp/vdisk.img
! rm -f /tmp/vdisk.img
! echo ""
! echo "完整挂载流程："
! echo "  sudo mkdir -p /mnt/vdisk"
! echo "  sudo mount -o loop /tmp/vdisk.img /mnt/vdisk"
! echo "  df -h /mnt/vdisk"
! echo "  sudo umount /mnt/vdisk"

In [ ]:
# /etc/fstab — 开机自动挂载
! cat /etc/fstab 2>/dev/null || echo "fstab 不可读"
! echo ""
! echo "fstab 每列含义："
! echo "  第1列: 设备或 UUID"
! echo "  第2列: 挂载点"
! echo "  第3列: 文件系统类型"
! echo "  第4列: 挂载选项（defaults,noatime）"
! echo "  第5列: dump 备份标志"
! echo "  第6列: fsck 顺序"
! echo ""
! echo "LVM 命令速查："
! echo "  pvcreate /dev/sdb1"
! echo "  vgcreate vg_data /dev/sdb1"
! echo "  lvcreate -L 10G -n lv_mysql vg_data"
! echo "  mkfs.ext4 /dev/vg_data/lv_mysql"
! echo "  mount /dev/vg_data/lv_mysql /var/lib/mysql"
! echo "  lvextend -L +5G /dev/vg_data/lv_mysql  # 在线扩容！"

---

## Day 13：包管理深入

### apt / dpkg 生态系统

| 层级 | 工具 | 用途 |
|------|------|------|
| 底层 | `dpkg` | 操作 .deb 文件本身 |
| 高层 | `apt` | 自动处理依赖，从仓库下载 |

### 高级技巧

```bash
apt-mark hold nginx          # 固定版本，防止意外升级
apt-mark unhold nginx        # 取消固定
apt-cache depends nginx      # 查看依赖树
apt-cache rdepends python3   # 谁依赖于 python3
dpkg -S /bin/ls              # 文件属于哪个包
dpkg -L bash                 # 包安装了哪些文件
apt-get clean                # 清理缓存
apt-get autoremove           # 删除不再需要的依赖包
```

### 从源码编译安装

标准三步曲：`./configure --prefix=/usr/local` → `make -j$(nproc)` → `sudo make install`

何时需要编译安装：
1. 官方包太旧，需要新版本
2. 需要自定义编译选项
3. 想了解软件的内部结构

In [ ]:
# 包管理查询演示
! cat /etc/os-release 2>/dev/null | head -2 || lsb_release -a 2>/dev/null
! echo ""
! if command -v dpkg &>/dev/null; then
!     echo "已安装包数量: $(dpkg -l 2>/dev/null | wc -l)"
! fi
! echo ""
! echo "仓库配置："
! ls /etc/apt/sources.list.d/ 2>/dev/null || echo "无第三方仓库"
! echo ""
! echo "编译安装示例（演示流程，不执行）："
! echo "  git clone https://github.com/sharkdp/bat.git"
! echo "  cd bat"
! echo "  cargo build --release"
! echo "  sudo cp target/release/bat /usr/local/bin/"

---

## Day 14：第二周综合练习

### 搭建一个标准化的应用服务器环境

**任务 1：用户管理**
```
创建三个用户：
  - deployer: 可 sudo 重启特定服务，有 SSH 登录权限
  - webapp:   运行应用的系统用户，不可登录
  - database: 运行数据库的系统用户，不可登录
```

**任务 2：PAM 安全配置**
```
部署 SSH 登录失败 5 次后锁定 15 分钟
```

**任务 3：存储管理**
```
创建一个 500MB LVM 逻辑卷（或回环设备模拟），挂载到 /data
配置 fstab 确保开机自动挂载
```

**任务 4：Systemd Service**
```
为 webapp 用户写一个 systemd service：
  - 名称: webapp.service
  - 类型: simple
  - 依赖 /data 挂载点和网络
  - 失败后自动重启
```

**任务 5：Systemd Timer**
```
每天凌晨 2:00 运行数据库备份脚本
  - 错过则开机补跑
```

**任务 6：日志配置**
```
journald 持久化存储，限制最大 500MB，保留最多 14 天
```

In [ ]:
# 第二周检查清单
! cat << 'CHECKLIST'
# 第二周综合练习检查表

## 任务1：用户创建
[ ] sudo useradd -r -s /usr/sbin/nologin -M webapp
[ ] sudo useradd -r -s /usr/sbin/nologin -M database
[ ] sudo useradd -m -s /bin/bash deployer

## 任务2：PAM 配置
[ ] /etc/pam.d/sshd 中配置 pam_tally2.so deny=5 unlock_time=900

## 任务3：LVM/存储
[ ] lvcreate -L 500M -n lv_data vg0
[ ] mkfs.ext4 /dev/vg0/lv_data
[ ] mount /dev/vg0/lv_data /data
[ ] /etc/fstab 已配置 UUID

## 任务4：Service
[ ] /etc/systemd/system/webapp.service 已创建
[ ] sudo systemctl enable --now webapp

## 任务5：Timer
[ ] /etc/systemd/system/db-backup.{service,timer}
[ ] sudo systemctl enable --now db-backup.timer

## 任务6：Journald
[ ] Storage=persistent
[ ] SystemMaxUse=500M
[ ] MaxRetentionSec=14day
CHECKLIST

---

## 第2周总结

| 概念 | 一句话 | 关键命令 |
|------|--------|----------|
| **systemd** | Linux 的 PID 1，管理系统服务 | `systemctl`, `journalctl` |
| **Timer** | 现代化 cron | `OnCalendar`, `Persistent=true` |
| **PAM** | 可插拔认证框架 | `/etc/pam.d/`, `pam_tally2.so` |
| **LVM** | PV→VG→LV 灵活存储管理 | `lvcreate`, `lvextend` |
| **fstab** | 开机自动挂载配置 | `UUID=xxx /data ext4 defaults 0 2` |
| **apt-mark hold** | 固定包版本 | |

### 本周命令肌肉记忆

```bash
systemctl status nginx           # 查看服务状态
journalctl -u nginx -f           # 跟踪服务日志
journalctl --vacuum-size=500M    # 控制日志大小
useradd -r -s /usr/sbin/nologin  # 创建系统用户
visudo                           # 安全编辑 sudo
lsblk -f                         # 查看块设备和 UUID
dpkg -S /bin/ls                  # 查文件属于哪个包
```